In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_replace, max

In [ ]:
spark = SparkSession.builder \
    .master('local[*]') \
    .appName("ex-1") \
    .getOrCreate()

In [ ]:
listings = spark.read.csv(
    "../data-src/listings.csv.gz",
    header=True,
    inferSchema=True,
    quote='"',
    escape='"',
    sep=",",
    multiLine=True,
    mode="PERMISSIVE"
)

listings.show(5)

In [ ]:
# Q1: Get a non-null picture URL for any property
listings.filter(col("picture_url").isNotNull()) \
    .select("picture_url") \
    .limit(1) \
    .show(truncate=False)

In [ ]:
# Q2: Get number of properties that get more than 10 reviews per month
listings \
    .filter('reviews_per_month > 10').count()

In [ ]:
# Q3: Get a property that has more bathrooms than bedrooms
more_bathrooms = listings.filter(col("bathrooms") > col("bedrooms")) \
    .select("id", "price", "source", "name", "bathrooms", "bedrooms") \
    .limit(1)

more_bathrooms.show(1)

In [ ]:
# Q4: Get 10 properties where the price is greater than 5000. Return the result as 
# a python list
price_num_df = listings \
    .withColumn("price_num", regexp_replace("price", "[$,]", "").cast("float")) \
    .filter("price_num > 5000") \
    .select("id", "name", "price", "price_num")

price_num_df.take(10)

In [ ]:
# Q5: Get the highest listing price in the dataset
max_price = price_num_df.select(max('price_num').alias("max_price")).collect()
max_price = max_price[0]['max_price']
max_price

In [ ]:
price_num_df.sort('price_num', ascending=False).show(10, truncate=False)

In [ ]:
price_num_df.select(max('price_num').alias("max_price")).collect()

In [ ]:
price_num_df.select(max('price_num').alias("max_price"))

In [ ]:
price_num_df.groupby('name').agg({"price_num": "sum"}) \
    .sort("name", ascending=False) \
    .show(truncate=False)

In [ ]:
spark.stop()